# Portfolio Replication — Full Comparison
Each section calls a single function from `utils/`. All outputs (figures, pickles) are saved automatically.

**Run order:** Data Loader → Linear Models → Kalman → NN → Evaluate

In [1]:
import sys, pickle, logging
from pathlib import Path

# ── path so imports resolve from the project root ────────────────────────
ROOT = Path('..').resolve()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from utils import setup_logging
setup_logging(logging.INFO)

PICKLE_DIR = ROOT / 'data' / 'picklefiles'


## 1 · Data Loader
Loads prices, computes weekly returns, constructs the **monster index** (50% HFRX, 25% MSCI World, 25% Global Agg Bond), and saves 5 diagnostic plots.

In [2]:
from utils.data_loader import run_data_loader

data = run_data_loader(
    filepath=ROOT / 'data' / 'Dataset3_PortfolioReplicaStrategy.xlsx'
)


14:41:47 | INFO     | utils.data_loader         — ============================================================
14:41:47 | INFO     | utils.data_loader         — DATA LOADER — START
14:41:47 | INFO     | utils.data_loader         — ============================================================
14:41:47 | INFO     | utils.data_loader         — Reading Excel file: C:\Users\giuli\Repositories\fintech-group-work\BusinessCase3\data\Dataset3_PortfolioReplicaStrategy.xlsx (sheet=0)
14:41:47 | INFO     | utils.data_loader         — Loaded 705 observations | 2007-10-23 → 2021-04-20
14:41:47 | INFO     | utils.data_loader         — Computing weekly returns from price levels
14:41:47 | INFO     | utils.data_loader         — Returns shape: (704, 15)
14:41:47 | INFO     | utils.data_loader         — Monster index composition: HFRXGL Index=50% | MXWO Index=25% | LEGATRUU Index=25%
14:41:47 | INFO     | utils.data_loader         — X shape: (704, 11) | y shape: (704,)
14:41:47 | INFO     | utils.data_loa

In [3]:
# Quick sanity check — shape and date range
print('Prices  :', data['prices'].shape,  '|', data['prices'].index[0].date(), '→', data['prices'].index[-1].date())
print('Returns :', data['returns'].shape)
print('X       :', data['X'].shape,  '(futures features)')
print('y       :', data['y'].shape,  '(monster index)')
data['y'].describe().round(5)


Prices  : (705, 15) | 2007-10-23 → 2021-04-20
Returns : (704, 15)
X       : (704, 11) (futures features)
y       : (704,) (monster index)


count    704.00000
mean       0.00049
std        0.00878
min       -0.06694
25%       -0.00361
50%        0.00143
75%        0.00552
max        0.03377
Name: MonsterIndex, dtype: float64

*Plots saved to `outputs/01_price_series.png` … `05_return_stats.png`.*

## 2 · Linear Models (OLS / Ridge / LASSO / Elastic Net)
> **TODO** — `utils/run_linear_models.py` not yet created.

Once available, this cell will call:
```python
from utils.run_linear_models import run_linear_models
linear_results = run_linear_models(data['X'], data['y'])
```

In [4]:
# Load pre-computed linear model results if the pickle exists
linear_pkl = PICKLE_DIR / 'linear_results.pkl'
if linear_pkl.exists():
    with open(linear_pkl, 'rb') as fh:
        linear_data = pickle.load(fh)
    linear_results = linear_data['best_results']   # dict: model_name → result_dict
    print('Loaded linear model results:', list(linear_results.keys()))
else:
    linear_results = {}
    print('No linear model pickle found — skipping.')


No linear model pickle found — skipping.


## 3 · Kalman Filter
> **TODO** — `utils/run_kalman.py` not yet created.

Once available:
```python
from utils.run_kalman import run_kalman
kalman_results = run_kalman(data['X'], data['y'])
```

In [5]:
kalman_pkl = PICKLE_DIR / 'kalman_results.pkl'
if kalman_pkl.exists():
    with open(kalman_pkl, 'rb') as fh:
        kalman_data = pickle.load(fh)
    kalman_result = kalman_data['best_result']
    print('Loaded Kalman results')
else:
    kalman_result = None
    print('No Kalman pickle found — skipping.')


No Kalman pickle found — skipping.


## 4 · Neural Network
Trains MLP and LSTM weight-generator networks across multiple hyperparameter configurations. Evaluated on the held-out test set (last 25 % of data).

**Loss**: MSE(replica, target) + λ·L1(weights)  
**Post-processing**: VaR scaling to respect UCITS 1M VaR(99%) ≤ 20%

In [6]:
from utils.run_nn import run_nn, DEFAULT_CONFIGS

nn_output = run_nn(
    X=data['X'],
    y=data['y'],
    configs=DEFAULT_CONFIGS,   # or pass a custom list
    train_frac=0.60,
    val_frac=0.15,
    max_var_threshold=0.20,
)


14:41:50 | INFO     | utils.run_nn              — ============================================================
14:41:50 | INFO     | utils.run_nn              — NN EXPERIMENT — START
14:41:50 | INFO     | utils.run_nn              — ============================================================
14:41:50 | INFO     | utils.run_nn              — Data split: train=422 | val=106 | test=176 (total=704)
14:41:50 | INFO     | utils.run_nn              — --------------------------------------------------
14:41:50 | INFO     | utils.run_nn              — Config 1/4: MLP_w26_h64x32_l10.0
14:41:50 | INFO     | utils.run_nn              — Device: cpu
14:41:52 | INFO     | utils.run_nn              — Epoch    1/300 | train=0.000035 | val=0.000012 | patience 0/30
14:41:53 | INFO     | utils.run_nn              — Epoch   25/300 | train=0.000011 | val=0.000009 | patience 17/30
14:41:53 | INFO     | utils.run_nn              — Early stopping triggered at epoch 38
14:41:53 | INFO     | utils.run_nn       

In [7]:
# Per-config metrics summary
nn_output['metrics_df'][[
    'tracking_error', 'information_ratio',
    'correlation', 'sharpe_replica', 'max_dd_replica'
]].round(4)


,tracking_error,information_ratio,correlation,sharpe_replica,max_dd_replica
model,,,,,
MLP_w26_h64x32_l10.0,0.0363,-0.2377,0.8423,0.8412,0.1003
MLP_w52_h64x32_l10.001,0.0512,-0.5273,0.6865,0.9192,0.0476
MLP_w26_h128x64x32_l10.001,0.0502,-0.5090,0.6826,0.8373,0.0567
LSTM_w52_h64_l10.0,0.0344,-0.2271,0.8616,0.8903,0.0955


*Training curves, weight plots, and VaR charts saved to `outputs/nn_cfg*.png`.*

## 5 · Full Comparison
Aggregates all available model results into a single evaluation run. Generates 6 comparison plots and a metrics heatmap.

In [8]:
from utils.evaluation import run_evaluation

# Build the combined results dict — add models as they become available
all_results = {}

# Linear models
#all_results.update(linear_results)

# Kalman
#if kalman_result is not None:
#all_results['Kalman'] = kalman_result

# Neural network (best config only — add all if you want)
all_results['NN_best'] = nn_output['best_result']

print('Models being compared:', list(all_results.keys()))


Models being compared: ['NN_best']


In [9]:
metrics = run_evaluation(all_results, save_prefix='final')
metrics[[
    'tracking_error', 'information_ratio',
    'correlation', 'sharpe_replica', 'max_dd_replica'
]].round(4)


14:42:12 | INFO     | utils.evaluation          — ============================================================
14:42:12 | INFO     | utils.evaluation          — EVALUATION — START  (1 models)
14:42:12 | INFO     | utils.evaluation          — ============================================================
14:42:12 | INFO     | utils.evaluation          — 
         tracking_error  information_ratio  correlation  sharpe_replica
model                                                                  
NN_best          0.0344            -0.2271       0.8616          0.8903
14:42:13 | INFO     | utils.evaluation          — Figure saved → outputs\final_01_cumulative_returns.png
14:42:13 | INFO     | utils.evaluation          — Figure saved → outputs\final_02_tracking_metrics.png
14:42:13 | INFO     | utils.evaluation          — Figure saved → outputs\final_03_drawdowns.png
14:42:13 | INFO     | utils.evaluation          — Figure saved → outputs\final_04_scatter_returns.png
14:42:14 | INFO     | ut

c:\Users\giuli\Repositories\fintech-group-work\BusinessCase3\.venv\Lib\site-packages\seaborn\matrix.py:202: RuntimeWarning: All-NaN slice encountered
  vmin = np.nanmin(calc_data)
c:\Users\giuli\Repositories\fintech-group-work\BusinessCase3\.venv\Lib\site-packages\seaborn\matrix.py:207: RuntimeWarning: All-NaN slice encountered
  vmax = np.nanmax(calc_data)


14:42:14 | INFO     | utils.evaluation          — Pickle saved → data\picklefiles\evaluation.pkl
14:42:14 | INFO     | utils.evaluation          — EVALUATION — DONE
14:42:14 | INFO     | utils.evaluation          — ============================================================


,tracking_error,information_ratio,correlation,sharpe_replica,max_dd_replica
model,,,,,
NN_best,0.0344,-0.2271,0.8616,0.8903,0.0955


*Comparison plots saved to `outputs/final_01_*.png` … `final_06_*.png`.*

## 6 · Transaction Cost Analysis
Applies one-way cost scenarios of **0, 2, 5, 10 bps** to every model 
that exposes a `weights_history`. Generates turnover plots, cost-drag 
bars, IR-vs-cost curves, and a net cumulative returns chart.

In [10]:
from utils.transaction_costs import run_transaction_cost_analysis
from utils.evaluation import run_evaluation_with_costs

# ── Run cost analysis ────────────────────────────────────────────────────
cost_tables = run_transaction_cost_analysis(
    all_results,
    cost_scenarios=[0.0, 2.0, 5.0, 10.0],
    save_prefix='tc',
)


14:42:14 | INFO     | utils.transaction_costs   — ============================================================
14:42:14 | INFO     | utils.transaction_costs   — TRANSACTION COST ANALYSIS — START  (1 models)
14:42:14 | INFO     | utils.transaction_costs   — ============================================================
14:42:14 | INFO     | utils.transaction_costs   — Computing cost scenarios for 'NN_best' …
14:42:14 | INFO     | utils.transaction_costs   — Summary at 5 bps:
14:42:14 | INFO     | utils.transaction_costs   —   NN_best                        | net IR=-0.2597 | drag=0.1117% | mean TO=0.0430
14:42:14 | INFO     | utils.transaction_costs   — Figure saved → outputs\tc_01_turnover.png
14:42:14 | INFO     | utils.transaction_costs   — Figure saved → outputs\tc_02_cost_drag.png
14:42:15 | INFO     | utils.transaction_costs   — Figure saved → outputs\tc_03_ir_vs_cost.png
14:42:15 | INFO     | utils.transaction_costs   — Figure saved → outputs\tc_04_net_cumulative.png
14:42:15 | INF

In [11]:
# ── Combined gross + net metrics table (at 5 bps) ────────────────────────
combined_metrics = run_evaluation_with_costs(
    all_results,
    cost_bps=5.0,
    save_prefix='final_with_costs',
)

combined_metrics[[
    'tracking_error', 'information_ratio',
    'net_te', 'net_ir',
    'sharpe_replica', 'net_sharpe',
]].round(4)


14:42:15 | INFO     | utils.evaluation          — ============================================================
14:42:15 | INFO     | utils.evaluation          — EVALUATION — START  (1 models)
14:42:15 | INFO     | utils.evaluation          — ============================================================
14:42:15 | INFO     | utils.evaluation          — 
         tracking_error  information_ratio  correlation  sharpe_replica
model                                                                  
NN_best          0.0344            -0.2271       0.8616          0.8903
14:42:15 | INFO     | utils.evaluation          — Figure saved → outputs\final_with_costs_01_cumulative_returns.png
14:42:16 | INFO     | utils.evaluation          — Figure saved → outputs\final_with_costs_02_tracking_metrics.png
14:42:16 | INFO     | utils.evaluation          — Figure saved → outputs\final_with_costs_03_drawdowns.png
14:42:16 | INFO     | utils.evaluation          — Figure saved → outputs\final_with_costs_04_

c:\Users\giuli\Repositories\fintech-group-work\BusinessCase3\.venv\Lib\site-packages\seaborn\matrix.py:202: RuntimeWarning: All-NaN slice encountered
  vmin = np.nanmin(calc_data)
c:\Users\giuli\Repositories\fintech-group-work\BusinessCase3\.venv\Lib\site-packages\seaborn\matrix.py:207: RuntimeWarning: All-NaN slice encountered
  vmax = np.nanmax(calc_data)


14:42:17 | INFO     | utils.evaluation          — Pickle saved → data\picklefiles\evaluation.pkl
14:42:17 | INFO     | utils.evaluation          — EVALUATION — DONE
14:42:17 | INFO     | utils.evaluation          — ============================================================
14:42:17 | INFO     | utils.evaluation          — NN_best                        | net IR=-0.2597 | net TE=0.0344 | cost=5 bps
14:42:17 | INFO     | utils.evaluation          — Pickle saved → data\picklefiles\evaluation_with_costs.pkl


,tracking_error,information_ratio,net_te,net_ir,sharpe_replica,net_sharpe
model,,,,,,
NN_best,0.0344,-0.2271,0.0344,-0.2597,0.8903,0.8699


### Interpretation

Fill this section after running. Suggested structure:
- Which model has the highest **turnover** — and why?
- At 5 bps, how much does the IR degrade vs gross for each model?
- Is there a **crossover point** where a lower-TE model becomes 
preferable once costs are accounted for?
- Does any model's IR turn *more* negative than expected, suggesting 
excessive rebalancing?


## 6 · Interpretation

Fill this section with your own analysis after running the full pipeline.

Suggested structure:
- Which model achieves the lowest **Tracking Error**?
- Which has the highest **Information Ratio**?
- How does **gross exposure** compare across models (leverage)?
- How does the NN perform vs Kalman — does extra complexity pay off?
- Where does each model fail (high drawdown periods, regime changes)?
